# **Volatility-Managed Factor Investing: Forecasting versus Portfolio Performance**

**Research question.** Do more sophisticated volatility forecasts produce better volatility-managed factor portfolios once realistic transaction costs are taken into account?

Volatility-managed portfolios scale exposure to a factor by the inverse of a forecast of its variance, taking less risk when volatility is expected to be high. Moreira and Muir (2017) show that this simple rule raises Sharpe ratios and earns positive alphas across a wide range of equity factors, because changes in volatility are not offset by proportional changes in expected returns. A natural question follows: if the strategy depends on a *forecast* of variance, does a better forecast produce a better portfolio?

This notebook answers that question for the six Fama and French factors (Mkt, SMB, HML, RMW, CMA, and Mom) over a common Jan 1963 – Apr 2026 sample. We compare three variance forecasts: a random-walk benchmark (RV), the heterogeneous autoregressive model of Corsi (2009) (HAR), and a random forest (RF). For each forecast we build the Moreira and Muir (2017) volatility-managed portfolio, net out transaction costs, and evaluate both forecast accuracy and portfolio performance with formal statistical tests.

It turns out that more sophisticated forecasts are sometimes significantly more accurate, but this accuracy does not translate into net portfolio performance. In one case, the more accurate forecast even produces a significantly worse portfolio. Performance is driven far more by which factor is managed than by which forecast is used. This aligns with the findings from Cederburg et al. (2020), who find that out-of-sample volatility-managed strategies generally fail to beat the unmanaged factors once implementation is taken seriously.

---

## **References**

- Campbell, J. Y. and Thompson, S. B. (2008). Predicting excess stock returns out of sample: Can anything beat the historical average? *Review of Financial Studies* 21(4), 1509–1531.
- Cederburg, S., O'Doherty, M. S., Wang, F. and Yan, X. S. (2020). On the performance of volatility-managed portfolios. *Journal of Financial Economics* 138(1), 95–117.
- Corsi, F. (2009). A simple approximate long-memory model of realized volatility. *Journal of Financial Econometrics* 7(2), 174–196.
- Diebold, F. X. and Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business and Economic Statistics* 13(3), 253–263.
- Harvey, D., Leybourne, S. and Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting* 13(2), 281–291.
- Moreira, A. and Muir, T. (2017). Volatility-managed portfolios. *Journal of Finance* 72(4), 1611–1644.
- Patton, A. J. (2011). Volatility forecast comparison using imperfect volatility proxies. *Journal of Econometrics* 160(1), 246–256.
- Politis, D. N. and Romano, J. P. (1994). The stationary bootstrap. *Journal of the American Statistical Association* 89(428), 1303–1313.


## 1. **Setup**

All analysis logic is stored in the `src` folder. This notebook walks through it. We add `src` to the path and import the pipeline functions.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd

from data import load_factor_data
from features import create_volatility_features, factors
from forecasting import get_forecasts
from portfolio import construct_vol_managed_returns
from backtest import run_backtest
from evaluation import (
    models,
    forecast_accuracy_table,
    performance_metrics,
    sharpe_ratio,
    sharpe_difference_table,
)
from plots import (
    plot_cumulative_returns,
    plot_drawdowns,
    plot_turnover,
    plot_forecast_accuracy,
)

pd.set_option("display.float_format", lambda v: f"{v:.3f}")

INITIAL_TRAIN_YEARS = 20
TCOST = 0.001
MAX_LEVERAGE = 3.0

## **2. Data**

We use daily and monthly returns for the six Fama and French factors, obtained from Kenneth French's data library. The five-factor set (Mkt, SMB, HML, RMW, CMA) begins in July 1963, which sets the common start date for all factors in our analysis. Momentum is appended from the separate momentum file. The market factor is the excess return $R_m - R_f$. All series are trimmed to the common window so that every factor shares an identical sample, and returns are stored in decimal form.

Daily returns are used to construct realised variance. Monthly returns are the series that is volatility-managed.

In [ ]:
daily_returns, monthly_returns = load_factor_data()

print("daily ", daily_returns.shape, daily_returns.index.min().date(), "to", daily_returns.index.max().date())
print("monthly", monthly_returns.shape, monthly_returns.index.min().date(), "to", monthly_returns.index.max().date())
monthly_returns.head()

## 3. **Volatility features**

As a proxy for the realised variance, we use the sum of squared daily returns. Daily realised variance is $RV_d = r_d^2$, and the $h$-day backward average is

$$RV_{d \mid d-h} = \frac{1}{h} \sum_{i=1}^{h} RV_{d-i+1}.$$

Because portfolios are formed monthly, we take one observation on the last trading day $d_t$ of each month and form the three Corsi (2009) components at the daily, weekly, and monthly horizons ($h = 1, 5, 22$). The forecast target is the forward 22-day realised variance,

$$RV_{d_t + 22 \mid d_t} = \sum_{j=1}^{22} RV_{d_t + j},$$

which is the variance the portfolio actually experiences over the following month. Measuring all regressors on day $d_t$ ensures there is no look-ahead bias.

In [ ]:
features = create_volatility_features(daily_returns, "Mkt")
features.head()

## **4. Volatility forecasts**

We compare three forecasts of next month's variance, all trained on the same features and target such that any difference in performance could be directly linked to the model's functional form.

**RV (random walk).** The benchmark forecast is the realised variance of the past 22 trading days, $\sum_{i=1}^{22} RV_{d_t - i + 1}$. Note that this does not require any estimation.

**HAR (Corsi, 2009).** A linear regression of the forward variance on the three components,

$$RV_{d_t + 22 \mid d_t} = \alpha + \beta^{(d)} RV_{d_t} + \beta^{(w)} RV_{d_t \mid d_t - 5} + \beta^{(m)} RV_{d_t \mid d_t - 22} + \varepsilon_t,$$

estimated by OLS. The three horizons capture volatility persistence within the month.

**RF (random forest).** A nonlinear ensemble on the same three features, allowing for interactions and threshold effects that the linear HAR cannot represent.

All models use an expanding window with an initial training period of 20 years and are refit each month. The random forest hyperparameters (`n_estimators`, `max_depth`, `min_samples_leaf`) are retuned annually on the most recent 20% of the training block as a validation sample. This sample is used only to select hyperparameters and is included in the training data when the fitted model produces forecasts. Forecasts are cached in `results/forecasts/`, so the cell below is fast on a second run.

In [ ]:
forecasts = {f: get_forecasts(daily_returns, f) for f in factors}
forecasts["Mkt"].head()

## **5. Forecast accuracy**

Forecast accuracy is the first half of the research question. We evaluate it with three measures.

**Out-of-sample $R^2$ (Campbell and Thompson, 2008).** Measured against the expanding historical mean of realised variance,

$$R^2_{oos} = 1 - \frac{\sum_t \left( RV_{t+1} - \widehat{RV}_{t+1} \right)^2}{\sum_t \left( RV_{t+1} - \overline{RV}_t \right)^2},$$

where $\overline{RV}_t$ is the mean of realised variance through month $t$. A positive value means the model beats the naive mean.

**QLIKE (Patton, 2011).** The loss

$$\text{QLIKE} = \frac{1}{T} \sum_t \left( \frac{RV_{t+1}}{\widehat{RV}_{t+1}} - \log \frac{RV_{t+1}}{\widehat{RV}_{t+1}} - 1 \right)$$

is robust to the fact that realised variance is a noisy proxy for true variance, and penalises under-prediction of risk more heavily than over-prediction. Lower is better.

**Diebold and Mariano (1995) test.** Tests whether the difference in QLIKE loss between two forecasts is significant, with the Harvey, Leybourne and Newbold (1997) small-sample correction for the one-step horizon.

In [ ]:
r2_rows = {}
qlike_rows = {}
dm_rows = {}
for factor in factors:
    realised = create_volatility_features(daily_returns, factor)["rv_fwd"].reindex(forecasts[factor].index)
    accuracy, dm = forecast_accuracy_table(forecasts[factor], realised)
    r2_rows[factor] = accuracy["r2_oos"]
    qlike_rows[factor] = accuracy["qlike"]
    dm_rows[factor] = dm

r2_table = pd.DataFrame(r2_rows).T[models]
qlike_table = pd.DataFrame(qlike_rows).T[models]
dm_table = pd.DataFrame(dm_rows).T

print("Out-of-sample R^2")
print(r2_table)
print("\nQLIKE loss")
print(qlike_table)
print("\nDiebold-Mariano p-values (QLIKE loss)")
print(dm_table)

In [ ]:
plot_forecast_accuracy(r2_table)

The accuracy ranking is factor-dependent rather than uniform. For HML and CMA all three models are close and strongly positive. Hence, variance is highly forecastable for these factors and more sophisticated models add little. For the market, HAR and RF clearly beat the random walk, whose $R^2$ is negative. For SMB only the random forest beats the naive mean. No single model dominates across factors, and the Diebold-Mariano tests confirm that the significant accuracy gains are concentrated in a few factor-model pairs.

## **6. Volatility-managed portfolios**

Following Moreira and Muir (2017), the volatility-managed weight is

$$w_{t+1} = \frac{c}{\widehat{\sigma}^2_{t+1}},$$

where $\widehat{\sigma}^2_{t+1}$ is the variance forecast formed at month $t$ and $c$ is a constant chosen so that the managed series has the same unconditional variance as the unmanaged factor. To avoid look-ahead bias, $c$ is estimated only on the initial training window and then held fixed. The weight is applied to the factor return realised in month $t+1$.

Two implementation constraints make the strategy realistic. First, leverage is capped at 3, which reflects the limited leverage a real investor can obtain. Without a cap, the rule can demand very high leverage that a real investor could not take. Second, every rebalance costs 10 basis points on the amount traded. Because the existing position drifts with the factor during the month, we only pay to trade the gap between the new target and the drifted weight, $\lvert w_{t+1} - w_t (1 + r_{t+1}) \rvert$, not the full position.

In [ ]:
results = {}
for factor in factors:
    results[factor] = {}
    for model in models:
        managed = construct_vol_managed_returns(
            monthly_returns[factor], forecasts[factor][model],
            train_years=INITIAL_TRAIN_YEARS, cap=MAX_LEVERAGE,
        )
        results[factor][model] = run_backtest(managed, tcost=TCOST)

results["Mkt"]["HAR"].head()

## **7. Portfolio performance**

For each of the eighteen portfolios we report the annualised net Sharpe ratio, the annualised geometric return, the maximum drawdown, the average monthly turnover, and the alpha from regressing the managed net return on the unmanaged factor. The alpha is the Moreira and Muir (2017) test of whether volatility management adds return compared to unmanaged. Because the factors are excess (or self-financing) returns, the Sharpe ratio is the annualised mean over the standard deviation.

In [ ]:
performance_rows = {}
sharpe_rows = {}
for factor in factors:
    for model in models:
        result = results[factor][model]
        performance_rows[(factor, model)] = performance_metrics(result, result["factor_return"])
        sharpe_rows.setdefault(factor, {})[model] = sharpe_ratio(result["net_return"])

sharpe_table = pd.DataFrame(sharpe_rows).T[models]
performance_table = pd.DataFrame(performance_rows).T

print("Net Sharpe ratio")
print(sharpe_table)

In [ ]:
performance_table

In [ ]:
plot_cumulative_returns(results)

In [ ]:
plot_drawdowns(results)

In [ ]:
plot_turnover(results)

Two patterns stand out. The spread in Sharpe ratios across factors is far larger than the spread across forecasts: managing momentum or profitability is very different from managing size, regardless of which forecast is used. And the turnover chart shows that HAR consistently trades least, because its forecasts are smoother than the random forest's. Trading less means paying less, which is how a simpler forecast can keep up with or beat a more accurate one when we find lower turnover.

## **8. Sharpe-difference tests**

Point estimates of Sharpe ratios are noisy, so we test whether the differences between forecasts are significant. We use the stationary bootstrap of Politis and Romano (1994). Specifically, we resample the two net-return series with the same block indices so that their contemporaneous dependence is preserved, and build the distribution of the Sharpe difference. The tables below are upper-triangular: each cell is the two-sided $p$-value for the row model against the column model.

In [ ]:
net_returns_by_factor = {
    factor: {model: results[factor][model]["net_return"] for model in models}
    for factor in factors
}

for factor in factors:
    print(factor)
    print(sharpe_difference_table(net_returns_by_factor[factor]))
    print()

The tests deliver a clear story. Most model differences are not significant. Where they are, the nonlinear random forest never significantly beats the random walk, and on profitability it is significantly worse. The only case where HAR significantly beats the random walk is size, a factor for which volatility management fails outright (all Sharpe ratios are negative), so this is a case of HAR being the least bad of three losing strategies. Hence, not a real win.

HAR is significantly more accurate than the random walk for the market, value, and investment factors, yet it does not significantly improve their net Sharpe ratios. The random forest is significantly more accurate than the random walk for profitability, yet it produces a significantly *worse* portfolio there. Better variance forecasts do not translate into better volatility-managed portfolios.

## **9. Conclusions**

- More sophisticated volatility forecasts do not produce better volatility-managed factor portfolios after transaction costs.
- Forecast accuracy and portfolio performance are disconnected: significant gains in out-of-sample $R^2$ or QLIKE do not map into significant gains in net Sharpe ratios, and for profitability the more accurate random forest yields a significantly worse portfolio.
- Performance is determined far more by which factor is managed than by which forecast is used. Volatility management clearly helps the momentum and profitability factors and clearly hurts the size factor.
- The mechanism is turnover. The smoother HAR forecast trades less and so pays lower costs. This allows a simpler model to match or beat a more accurate one once implementation is taken into account.
- These results are consistent with Cederburg et al. (2020): volatility management looks attractive in-sample, but an investor trading in real time, with costs and leverage limits, struggles to actually capture those gains.